# 04 - Smart Chunking & Metadata-First Retrieval

This notebook demonstrates the **structure-aware chunking** and **metadata-first retrieval** features:

1. Compare generic `SentenceSplitter` vs. `ContractNodeParser` on a sample contract
2. Inspect extracted structural metadata (`section_type`, `has_table`, `clause_number`)
3. Query with `section_type` filter (e.g. payment terms for a specific contract)
4. Query with `has_table` filter (e.g. find all pricing tables)
5. Compare filtered vs. unfiltered retrieval results

**Prerequisites:** Run `01_generate_data.ipynb` and `02_ingestion.ipynb` first.

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root (needed when running from notebooks/ directory)
load_dotenv(Path('../.env'))

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

settings = get_settings()
init_observability(settings)

## 1. Generic vs. Structure-Aware Chunking

Let's load a single contract and compare how `SentenceSplitter` and `ContractNodeParser` handle it.

In [2]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from contract_lens.ingestion.pipeline import parse_filename_metadata
from contract_lens.ingestion.node_parser import ContractNodeParser

# Load clean PDFs (agreements have extractable text; scans are images without OCR)
data_path = Path('../data/agreements')
reader = SimpleDirectoryReader(input_dir=str(data_path), recursive=False)
documents = reader.load_data()

# Enrich metadata
for doc in documents:
    filename = doc.metadata.get('file_name', '')
    meta = parse_filename_metadata(filename)
    doc.metadata.update(meta)

# Filter to ITSVC-001 base contract only
itsvc_docs = [d for d in documents if d.metadata.get('contract_id') == 'ITSVC-001' and d.metadata.get('source_type') == 'base']
print(f'ITSVC-001 base contract: {len(itsvc_docs)} pages')

ITSVC-001 base contract: 3 pages


In [3]:
# Generic chunking
generic_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
generic_nodes = generic_splitter.get_nodes_from_documents(itsvc_docs)

# Structure-aware chunking
contract_parser = ContractNodeParser(chunk_size=1024, chunk_overlap=128)
smart_nodes = contract_parser.get_nodes_from_documents(itsvc_docs)

print(f'Generic SentenceSplitter: {len(generic_nodes)} chunks')
print(f'ContractNodeParser:       {len(smart_nodes)} chunks')
print(f'\n--- Generic chunk 0 (first 200 chars) ---')
print(generic_nodes[0].text[:200])
print(f'\n--- Smart chunk 0 (first 200 chars) ---')
print(smart_nodes[0].text[:200])

Generic SentenceSplitter: 3 chunks
ContractNodeParser:       9 chunks

--- Generic chunk 0 (first 200 chars) ---
IT SERVICE AGREEMENT
IT SERVICE AGREEMENT
Contract ID: ITSVC-001
Effective Date: 2025-01-15
Version: 1 (Base Contract)
This  IT  Service  Agreement  (the  'Agreement')  is  entered  into  as  of  2025

--- Smart chunk 0 (first 200 chars) ---
IT SERVICE AGREEMENT
IT SERVICE AGREEMENT
Contract ID: ITSVC-001
Effective Date: 2025-01-15
Version: 1 (Base Contract)
This  IT  Service  Agreement  (the  'Agreement')  is  entered  into  as  of  2025


## 2. Inspect Structural Metadata

Each smart chunk carries `section_type`, `section_name`, `has_table`, and `clause_number`.

In [4]:
# Parse all documents
all_smart_nodes = contract_parser.get_nodes_from_documents(documents)

print(f'Total smart chunks: {len(all_smart_nodes)}')
print(f'\n{"section_type":<18} {"has_table":<10} {"clause":<8} {"section_name"}')
print('-' * 80)
for node in all_smart_nodes[:20]:
    m = node.metadata
    print(f'{m.get("section_type", ""):<18} {m.get("has_table", ""):<10} {m.get("clause_number", ""):<8} {m.get("section_name", "")[:50]}')

Total smart chunks: 68

section_type       has_table  clause   section_name
--------------------------------------------------------------------------------
general            false               
scope              false      1.1      1. Scope of Services
termination        false      2.1      2. Term and Termination
payment            false      3.1      3. Payment Terms
confidentiality    false      4.1      4. Confidentiality
liability          false      5.1      5. Liability
general            false               
general            false               
annex              true                ANNEX A - Pricing Schedule
general            false               AMENDMENT NO. 1 TO IT SERVICE AGREEMENT
general            false               AMENDMENT NO. 1
TO IT SERVICE AGREEMENT (ITSVC-001
annex              true                1. Revised Pricing (replaces Annex A)
general            false               2. Extended Team (amends clause 1.3)
general            false               3. Unch

In [5]:
# Distribution of section types
from collections import Counter

type_counts = Counter(n.metadata.get('section_type', 'unknown') for n in all_smart_nodes)
table_counts = Counter(n.metadata.get('has_table', 'unknown') for n in all_smart_nodes)

print('Section type distribution:')
for st, count in type_counts.most_common():
    print(f'  {st:<18} {count}')

print(f'\nChunks with tables: {table_counts.get("true", 0)}')
print(f'Chunks without:     {table_counts.get("false", 0)}')

Section type distribution:
  general            40
  annex              8
  payment            5
  sla                5
  scope              3
  termination        3
  penalties          2
  confidentiality    1
  liability          1

Chunks with tables: 7
Chunks without:     61


## 3. Metadata-Filtered Retrieval: section_type

Query with `section_type="payment"` to find only payment-related chunks.

In [6]:
from contract_lens.retrieval.query_engine import query_contracts

# Without filter
answer_unfiltered = query_contracts(
    settings,
    question='What are the current hourly rates for ITSVC-001?',
    contract_id='ITSVC-001',
)
print('=== WITHOUT section_type filter ===')
print(answer_unfiltered)

2026-03-05 12:38:51,377 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


2026-03-05 12:38:54,643 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


=== WITHOUT section_type filter ===
The current hourly rates (effective January 1, 2026) are:

- Senior Architect: **$310/hour**
- Cloud Engineer: **$260/hour**
- DevOps Specialist: **$235/hour**
- Project Manager: **$195/hour**
- QA Engineer: **$175/hour**
- AI/ML Engineer: **$320/hour**


In [7]:
# With section_type=payment filter
answer_filtered = query_contracts(
    settings,
    question='What are the current hourly rates for ITSVC-001?',
    contract_id='ITSVC-001',
    section_type='payment',
)
print('=== WITH section_type=payment filter ===')
print(answer_filtered)

2026-03-05 12:38:54,884 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


2026-03-05 12:38:57,599 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


=== WITH section_type=payment filter ===
The hourly rates for ITSVC-001 are set out in the pricing schedule in **Annex A**. The provided information does not include Annex A, so the current hourly rates cannot be determined from what’s available here.


## 4. Metadata-Filtered Retrieval: has_table

Find chunks that contain tables (pricing schedules, SLA metrics, etc.).

In [8]:
answer_tables = query_contracts(
    settings,
    question='What pricing tables exist across all contracts?',
    has_table=True,
)
print('=== has_table=true ===')
print(answer_tables)

2026-03-05 12:38:57,817 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


2026-03-05 12:39:03,437 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


=== has_table=true ===
Pricing tables found across the contracts:

- **ITSVC-001 (Amendment v3, effective 2026-01-01) — “Revised Pricing (replaces previous Annex A)”**  
  Columns: Service Category | Rate (USD/hour) | Min Hours/Month | Monthly Cap  
  Rows: Senior Architect; Cloud Engineer; DevOps Specialist; Project Manager; QA Engineer; AI/ML Engineer.

- **EMP-001 (Base v1, effective 2025-05-01) — “ZAŁĄCZNIK NR 1 - Wynagrodzenie i świadczenia”**  
  Columns: Składnik | Kwota/Opis | Częstotliwość  
  Rows include: Wynagrodzenie zasadnicze; Premia uznaniowa; Premia roczna; Pakiet medyczny; Karta sportowa; Budżet szkoleniowy; Sprzęt służbowy; Dodatek za pracę zdalną.


## 5. Combining Filters

Combine contract_id + section_type + source_type for precision retrieval.

In [9]:
# SLA penalty terms from amendments only
answer_sla_penalties = query_contracts(
    settings,
    question='What are the current SLA penalty terms and service credits?',
    contract_id='SLA-001',
    section_type='penalties',
)
print('=== SLA-001 penalties ===')
print(answer_sla_penalties)

2026-03-05 12:39:03,676 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


2026-03-05 12:39:07,449 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


=== SLA-001 penalties ===
- **Service credits entitlement (Clause 3.1):** If the Provider fails to meet the guaranteed service levels, the Client is entitled to service credits as specified in **Annex A**.  
- **Application of service credits (Clause 3.2):** Service credits are applied to the **next monthly invoice**.  
- **Monthly cap on service credits (amended Clause 3.3, effective 2025-10-01):** Total service credits in any calendar month shall not exceed **50% of that month’s fees**.


In [10]:
# Lease termination terms (PL)
answer_lease_term = query_contracts(
    settings,
    question='Jakie sa warunki wypowiedzenia umowy najmu LEASE-001?',
    contract_id='LEASE-001',
    section_type='termination',
    language='pl',
)
print('=== LEASE-001 termination (PL) ===')
print(answer_lease_term)

2026-03-05 12:39:07,673 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


2026-03-05 12:39:10,957 - INFO - HTTP Request: POST https://aihub-signalboy-shared-neu.cognitiveservices.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


=== LEASE-001 termination (PL) ===
Warunki wypowiedzenia umowy najmu LEASE-001 są następujące:

- **Wypowiedzenie przez każdą ze stron:** każda ze stron może wypowiedzieć umowę z **3‑miesięcznym okresem wypowiedzenia**.  
- **Wypowiedzenie przez Wynajmującego ze skutkiem natychmiastowym:** Wynajmujący może wypowiedzieć umowę **natychmiast**, jeśli wystąpią **zaległości w opłatach przekraczające 2 miesiące**.


## Summary

| Feature | Description |
|---------|-------------|
| `ContractNodeParser` | Splits at section boundaries, preserves clause context |
| `section_type` filter | Controlled vocabulary: scope, payment, termination, confidentiality, liability, sla, penalties, annex, general |
| `has_table` filter | Find chunks containing pricing tables, SLA metrics, etc. |
| `clause_number` filter | Target a specific clause (e.g. "3.1") |
| Combined filters | Stack multiple filters for precision retrieval |

See [docs/smart-chunking.md](../docs/smart-chunking.md) for full documentation.